so watch me recreate this fractional cover mapping of the whole area for Hawai'i
1st we have to find out the coordinates for our bounding box

In [ ]:
import geopandas as gpd

# Load the shapefile
gdf = gpd.read_file('/Users/gillopez00/Downloads/2020_Census_County_Boundaries/2020_Census_County_Boundaries.shp')
# Get the bounding box (minx, miny, maxx, maxy)
bounds = gdf.total_bounds
print(bounds)  # array([minx, miny, maxx, maxy])


minx, miny, maxx, maxy = bounds

[ 371245.5627 2094230.8512  940276.089  2458639.7979]


# Hashtag thank you to Tamara who helped me find out this coordinates for the bounding box of the islands of Hawai'i; basically what we did together was going to claude to ask it how I can get the coordinates from here in QGIS then, Claude gave me the code above ^ to do so 
All we had to do was figure out how to change the path file from the template to mine which tamara taught me by literally Command + C (on mac) basically and it produced the numbers for me 
[ 371245.5627 2094230.8512  940276.089  2458639.7979]
(West boundary, South boundary, East boundary, North boundary)

In [3]:
# Import Libraries
import os
import earthaccess
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from rioxarray.merge import merge_arrays
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

/Users/gillopez00/miniforge3/envs/emit-fcov/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# QC flag codes -- from singleband_raster_hierarchy() in create_frcov_masks.py,
# https://github.com/emit-sds/emit-sds-frcov (not formally published in the User Guide).
# Hierarchical: each pixel gets the FIRST condition below that applies to it.
flag_labels = {
    0: "clear",
    1: "cloud_or_cirrus",
    2: "urban",
    3: "water_or_coastal",
    4: "snow_ice",
}  # -9999 = nodata (outside scene footprint); handled separately via masked=True on load

flag_colors = ["#2ca02c", "#7f7f7f", "#d62728", "#1f77b4", "#f2f2f2"]
cmap = ListedColormap(flag_colors)
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap.N)

In [5]:
# Authenticate with NASA Earthdata (reads ~/.netrc, or prompts + persists on first run)
auth = earthaccess.login(persist=True)

In [6]:
# EMIT L2B Fractional Cover & Uncertainty (60 m, V001) over Maui.
# Currently narrowed to the exact acquisition date (2025-08-17) of the granule from
# https://search.earthdata.nasa.gov/...&g=G3991818701-LPCLOUD -- widen `temporal`
# (e.g. back to a month-long window) to search more broadly.
bbox = (-160.245911, 18.910690, -154.806622, 22.232707)  # (lon_min, lat_min, lon_max, lat_max)
temporal = ("2023-01-31", "2023-12-31")  # (start_date, end_date) in YYYY-MM-DD format

results = earthaccess.search_data(
    concept_id="C3911089796-LPCLOUD",  # EMIT_L2BFRCOV.001 collection (LPCLOUD)
    bounding_box=bbox,
    temporal=temporal,
)

print(f"Found {len(results)} granule(s)")

Found 47 granule(s)


/Users/gillopez00/miniforge3/envs/emit-fcov/lib/python3.12/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


#ok so i have to rework my first line of code to make sure that the coordinates are within the 0-180 format which I re-ran to Claude about and it gave me this instead to run

In [10]:
import geopandas as gpd

import earthaccess
earthaccess.login()

In [11]:
# Load the shapefile
gdf = gpd.read_file('/Users/gillopez00/Downloads/2020_Census_County_Boundaries/2020_Census_County_Boundaries.shp')

# Check the original CRS (useful to confirm what you're starting from)
print("Original CRS:", gdf.crs)

# Reproject to WGS84 (EPSG:4326) — the lat/lon system earthaccess expects
gdf_wgs84 = gdf.to_crs(epsg=4326)

# Get the bounding box in lon/lat
minx, miny, maxx, maxy = gdf_wgs84.total_bounds
print(f"lon_min: {minx:.6f}, lat_min: {miny:.6f}, lon_max: {maxx:.6f}, lat_max: {maxy:.6f}")

Original CRS: EPSG:3750
lon_min: -160.245911, lat_min: 18.910690, lon_max: -154.806622, lat_max: 22.232707


make sure to run the first 2 packages then this line of code that Claude gave me

Okay so sadly, this produced 0 granules... for the ENTIRE YEAR OF 2022
We tried going a year back (2021) but again 0 granules found... so moved forward to 2023 and found 47 granules!!! We will just rename this to 2023 then

In [7]:
# Print every file (granule ID + all associated data/browse links) found in the search
for granule in results:
    print(granule["meta"]["native-id"])
    for link in granule.data_links():
        print(f"  {link}")

EMIT_L2B_FRCOV_001_20230219T002729_2304916_002
  https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-public/EMITL2BFRCOV.001/EMIT_L2B_FRCOV_001_20230219T002729_2304916_002/EMIT_L2B_FRCOVQC_001_20230219T002729_2304916_002.tif
  https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-public/EMITL2BFRCOV.001/EMIT_L2B_FRCOV_001_20230219T002729_2304916_002/EMIT_L2B_FRCOVPV_001_20230219T002729_2304916_002.tif
  https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-public/EMITL2BFRCOV.001/EMIT_L2B_FRCOV_001_20230219T002729_2304916_002/EMIT_L2B_FRCOVPVUNC_001_20230219T002729_2304916_002.tif
  https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-public/EMITL2BFRCOV.001/EMIT_L2B_FRCOV_001_20230219T002729_2304916_002/EMIT_L2B_FRCOVNPV_001_20230219T002729_2304916_002.tif
  https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-public/EMITL2BFRCOV.001/EMIT_L2B_FRCOV_001_20230219T002729_2304916_002/EMIT_L2B_FRCOVNPVUNC_001_20230219T002729_2304916_002.tif
  https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-p

## Step 2: Locally download granules of interest

For every granule the search above found, there are 7 GeoTIFFs to download. Load them into one
 `xarray.Dataset`, and apply the QC band as a mask across the fraction
(PV/NPV/BARE) and uncertainty bands.

In [8]:
def download_and_reconcile(granule):
    """Download a granule's 7 GeoTIFFs, load into one aligned Dataset, mask
    fraction + uncertainty bands to QC-clear pixels (qc == 0). Returns (ds, ds_clear).
    QC flag source: create_frcov_masks.py in github.com/emit-sds/emit-sds-frcov."""
    granule_id = granule["meta"]["native-id"]
    out_dir = os.path.join("data", "emit_fcov", granule_id)
    os.makedirs(out_dir, exist_ok=True)
    files = earthaccess.download([granule], local_path=out_dir)

    roles = {}
    for f in files:
        name = os.path.basename(f)
        if "FRCOVQC" in name:
            roles["qc"] = f
        elif "FRCOVPVUNC" in name:
            roles["pv_unc"] = f
        elif "FRCOVNPVUNC" in name:
            roles["npv_unc"] = f
        elif "FRCOVBAREUNC" in name:
            roles["bare_unc"] = f
        elif "FRCOVPV" in name:
            roles["pv"] = f
        elif "FRCOVNPV" in name:
            roles["npv"] = f
        elif "FRCOVBARE" in name:
            roles["bare"] = f
    assert len(roles) == 7, f"Expected 7 bands, matched {len(roles)}: {roles}"

    g_ds = xr.Dataset(
        {role: rxr.open_rasterio(path, masked=True).squeeze("band", drop=True)
         for role, path in roles.items()}
    )

    clear = g_ds["qc"] == 0  # only unflagged pixels are retrieval-valid per the ATBD mask hierarchy
    g_ds_clear = g_ds.copy()
    for v in ["pv", "npv", "bare", "pv_unc", "npv_unc", "bare_unc"]:
        g_ds_clear[v] = g_ds[v].where(clear)

    return g_ds, g_ds_clear

In [9]:
# Process every granule the search found (not just one hardcoded scene) so this
# notebook still works if you change the bbox/temporal window above.
granules = {}
for g in results:
    gid = g["meta"]["native-id"]
    print(f"Processing {gid} ...")
    g_ds, g_ds_clear = download_and_reconcile(g)
    granules[gid] = (g_ds, g_ds_clear)

    total = int(g_ds["qc"].notnull().sum())
    n_clear = int((g_ds["qc"] == 0).sum())
    print(f"  {n_clear}/{total} pixels clear ({100 * n_clear / total:.1f}%)")

print(f"\nReconciled {len(granules)} granule(s).")

/Users/gillopez00/miniforge3/envs/emit-fcov/lib/python3.12/site-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


Processing EMIT_L2B_FRCOV_001_20230219T002729_2304916_002 ...


QUEUEING TASKS | : 100%|██████████| 7/7 [00:00<00:00, 2241.40it/s]
PROCESSING TASKS | : 100%|██████████| 7/7 [00:00<00:00, 44620.26it/s]
COLLECTING RESULTS | : 100%|██████████| 7/7 [00:00<00:00, 54674.35it/s]
/Users/gillopez00/miniforge3/envs/emit-fcov/lib/python3.12/site-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


  0/1995684 pixels clear (0.0%)
Processing EMIT_L2B_FRCOV_001_20230219T002741_2304916_003 ...


QUEUEING TASKS | : 100%|██████████| 7/7 [00:00<00:00, 4252.63it/s]
PROCESSING TASKS | : 100%|██████████| 7/7 [00:00<00:00, 73035.14it/s]
COLLECTING RESULTS | : 100%|██████████| 7/7 [00:00<00:00, 140479.08it/s]
/Users/gillopez00/miniforge3/envs/emit-fcov/lib/python3.12/site-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


...

  126781/1964126 pixels clear (6.5%)
Processing EMIT_L2B_FRCOV_001_20231001T213800_2327414_004 ...


QUEUEING TASKS | : 100%|██████████| 7/7 [00:00<00:00, 4028.00it/s]
PROCESSING TASKS | : 100%|██████████| 7/7 [00:00<00:00, 22883.97it/s]
COLLECTING RESULTS | : 100%|██████████| 7/7 [00:00<00:00, 86353.32it/s]
/Users/gillopez00/miniforge3/envs/emit-fcov/lib/python3.12/site-packages/earthaccess/store.py:838: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


  0/2663258 pixels clear (0.0%)
Processing EMIT_L2B_FRCOV_001_20231021T232441_2329415_003 ...


QUEUEING TASKS | : 100%|██████████| 7/7 [00:00<00:00, 2194.33it/s]
PROCESSING TASKS | : 100%|██████████| 7/7 [00:00<00:00, 68120.95it/s]
COLLECTING RESULTS | : 100%|██████████| 7/7 [00:00<00:00, 127100.12it/s]


  1463/1725471 pixels clear (0.1%)

Reconciled 47 granule(s).


# Mosaic All Scenes

Merge every granule's QC-masked Dataset into one continuous mosaic. Works for any number
of granules (1..N) — if the search above returns a single scene the "mosaic" is just
that scene; if it returns several, they all get merged into one.

In [10]:
mosaic_vars = ["qc", "pv", "npv", "bare", "pv_unc", "npv_unc", "bare_unc"]

ds_mosaic = xr.Dataset({
    v: merge_arrays([g_ds_clear[v] for _, g_ds_clear in granules.values()], nodata=np.nan)
    for v in mosaic_vars
})

print(f"Mosaicked {len(granules)} scene(s) -> shape {ds_mosaic['pv'].shape}")

frac_sum = ds_mosaic["pv"] + ds_mosaic["npv"] + ds_mosaic["bare"]
print(f"Mosaic PV+NPV+BARE (clear pixels) -> mean: {float(frac_sum.mean(skipna=True)):.4f}, "
      f"std: {float(frac_sum.std(skipna=True)):.4f} (should be ~1.0 / ~0)")

Mosaicked 47 scene(s) -> shape (9689, 14107)
Mosaic PV+NPV+BARE (clear pixels) -> mean: 1.0000, std: 0.0000 (should be ~1.0 / ~0)


In [1]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

im0 = ds_mosaic["qc"].plot.imshow(ax=axes[0], cmap=cmap, norm=norm, add_colorbar=False)
axes[0].set_title(f"Mosaic QC Flags ({len(granules)} scene(s))")
axes[0].set_aspect("equal")
cbar = fig.colorbar(im0, ax=axes[0], ticks=[0, 1, 2, 3, 4])
cbar.ax.set_yticklabels(list(flag_labels.values()))

rgb_mosaic = xr.concat(
    [ds_mosaic["bare"].fillna(0), ds_mosaic["pv"].fillna(0), ds_mosaic["npv"].fillna(0)],
    dim="band",
).transpose("y", "x", "band")

rgb_mosaic.plot.imshow(ax=axes[1], rgb="band")
axes[1].set_title("Mosaic Fractional Cover (R=BARE, G=PV, B=NPV)")
axes[1].set_aspect("equal")

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [12]:
print(ds_mosaic.nbytes / 1e9, "GB")
print(ds_mosaic.dims)

7.107691972 GB
FrozenMappingWarningOnValuesAccess({'x': 14107, 'y': 9689})


#basically this tells us how large the mosaic is; which is why the whole thing kills the kernel and crashes before it can produce any kind of plot
had to move it to 20 because it still crashed


In [11]:
factor = 20  # try 20 if this still crashes, or 5 if you want more detail

def prep(band):
    return ds_mosaic[band].fillna(0).coarsen(x=factor, y=factor, boundary="trim").mean()

rgb_mosaic = xr.concat([prep("bare"), prep("pv"), prep("npv")], dim="band").transpose("y", "x", "band")

qc_small = ds_mosaic["qc"].coarsen(x=factor, y=factor, boundary="trim").mean()

Claude ^ gave us the code above to help us coarsen the plots so that it doesn't crash; if it does change the factor to 20 for further coarsening 
so run this line of code first, then, run the plot of code

In [3]:
print(ds_mosaic.chunks)

NameError: name 'ds_mosaic' is not defined